# Day 5 Capstone — Website Maintenance Agent
A scheduler runs this once a day. It checks a source of updates, and if something new
appeared it drafts a change to a website and asks a human to approve it.

The scheduling part is ordinary automation. The interesting part is everything you built
this week: the two actions are **registered harness tools** with honest risk levels, so
drafting is allowed automatically and publishing is not.


## Before you begin

### Learning outcomes

- Register a real workflow's actions as harness tools and drive them from the runtime.
- Read the events that prove policy paused the run before anything was published.
- Watch change detection process a backlog and then report no change.

Architecture reference: [Day 5 diagrams D19](../diagrams/source/day_05.md).

### Expected observation

`propose_update` is allowed and `publish_update` pauses; the website file does not exist until an explicit approval; a poisoned source is refused by a guardrail and the refusal appears in the event log.


## Concept briefing

## Automation is a trigger, not intelligence

A scheduler can start a run every day, but scheduling alone is ordinary automation. The
agentic decision is whether new evidence warrants a change and which permitted action to
propose. Policy then decides whether the exact proposal may proceed.

The Website Maintenance Agent demonstrates a production-shaped cycle at classroom scale:
fetch a real or cached public update, compare it with durable processed-item state, create
a structured website proposal, apply guardrails, pause for approval, write a real local
file, verify the result and record events. The scheduler should call one bounded `check`
operation; it should not contain hidden business logic.

An optional LLM judge may score whether the proposed update is faithful to its source.
That judge belongs after deterministic checks and before approval or publication. It is
advisory because it can be inconsistent, biased toward fluent text or influenced by the
content it evaluates. File-path, schema, source, build and permission checks remain
authoritative application code.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/mini_harness"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Day 5 helper: agent configurations are DATA. Read one from configs/<name>.json
#    and turn it into the AgentConfig dataclass the runtime expects.
import json
from mini_harness import AgentConfig, ModelConfig

def load_config(name):
    raw = json.loads((PROJECT_ROOT / "configs" / f"{name}.json").read_text(encoding="utf-8"))
    raw["model"] = ModelConfig(**raw["model"])   # nested dict -> nested dataclass
    return AgentConfig(**raw)

print("Configs      :", sorted(p.stem for p in (PROJECT_ROOT / "configs").glob("*.json")))

## Step 1 — A fresh folder for this run

Everything is written under `data/generated/`, in a folder named for this moment. Re-run
the notebook as often as you like: each run starts from an empty website and empty state.


In [ ]:
from datetime import datetime
from uuid import uuid4

RUN_ROOT = (PROJECT_ROOT / "data" / "generated" /
            f"website_{datetime.now():%Y%m%d_%H%M%S}_{uuid4().hex[:6]}")
SITE_ROOT = RUN_ROOT / "site"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print("Run folder   :", RUN_ROOT.name)
print("Website root :", SITE_ROOT)
print("Fresh start  : site folder exists?", SITE_ROOT.exists())

## Step 2 — The source, and what "new" means

The cached source is a small JSON file so the lesson is repeatable. Durable state records
which items have already been handled; anything not in that list is new work.


In [ ]:
from mini_harness import CachedJSONSource, JSONStateStore

source = CachedJSONSource(PROJECT_ROOT / "data" / "website_updates.json")
state = JSONStateStore(RUN_ROOT / "state.json")

items = source.fetch()
print("Items available from the source:", len(items))
for item in items:
    print(f"  {item.item_id:<20} {item.title}")
    print(f"  {'':<20} from {item.url}")

current = state.load()
print()
print("Already processed:", current["processed_ids"] or "nothing yet (fresh state)")
unseen = [i for i in items if i.item_id not in current["processed_ids"]]
print("Therefore new    :", [i.item_id for i in unseen])

## Step 3 — Guardrails are the domain expert

Policy answers "may this *kind* of action happen?". Guardrails answer the questions only
this application can: is that host trusted, does the text contain instructions aimed at
the model, does the target path stay inside the website folder?


In [ ]:
from mini_harness import WebsiteGuardrails

guardrails = WebsiteGuardrails(SITE_ROOT, trusted_hosts={"github.com"})

print("Website root guarded  :", guardrails.site_root.name)
print("Trusted source hosts  :", guardrails.trusted_hosts)
print("Maximum body length   :", guardrails.max_body_chars, "characters")
print()
clean = unseen[0]
print("Input guardrail on", clean.item_id, "->", guardrails.check_source(clean) or "no failures")

## Step 4 — The two actions, registered as harness tools

This is the step that connects the capstone to the rest of Day 5. Drafting is reversible
and local, so it is `write`. Publishing changes the website, so it is `external`.


In [ ]:
from mini_harness import build_website_registry, deterministic_proposer, website_agent_config

# deterministic_proposer is honest about itself: it copies the cleaned-up source
# summary verbatim. There is NO model call on this path - which is exactly why the
# whole capstone runs with no API key. Step 8 swaps in the live proposer.
proposer = deterministic_proposer

registry = build_website_registry(source, proposer, guardrails, state)
config = website_agent_config()

print("Registered website tools:")
for spec in registry.discover():
    print(f"  {spec.name:<16} risk={spec.risk:<10} {spec.description}")
print()
print("Agent configuration:", config.name)
print("  allowed tools:", config.allowed_tools)
print("  plan          :", [step["tool"] for step in config.mock_plan])

In [ ]:
from mini_harness import decide

print("What policy will say before anything runs:")
for spec in registry.discover():
    print(f"  {spec.name:<16} -> {decide(config, spec)}")
print()
print("Nothing here is a special case. It is the same policy.decide from Day 5.4,")
print("reading the same risk levels, for a workflow that touches a real file.")

## Step 5 — One scheduled tick

The runtime drives both tools. Watch where it stops.


In [ ]:
from mini_harness import EventStore, HarnessRuntime, JSONCheckpointStore, MockModel

events = EventStore(RUN_ROOT / "events.jsonl")
runtime = HarnessRuntime(registry, MockModel(), events,
                         JSONCheckpointStore(RUN_ROOT / "checkpoints"))

target = SITE_ROOT / "content" / "updates.md"
print("Website file before the run:", "exists" if target.exists() else "does not exist")

result = runtime.run(config, clean.item_id)
print("Run status:", result.status)
print()
for event in result.events:
    d = event["details"]
    note = d.get("decision") or d.get("tool") or ""
    print(f"  {event['event']:<20} {note}")

In [ ]:
# The events are the evidence. Read them as a sequence of decisions.
print("Policy decisions, in order:")
for event in result.events:
    if event["event"] == "policy_decision":
        d = event["details"]
        print(f"  {d['tool']:<16} risk={d['risk']:<10} -> {d['decision']}")

print()
print("Paused on           :", result.pending_action["tool"])
print("Arguments held      :", result.pending_action["arguments"])
print("Website file exists :", target.exists(), "  <- observe: still nothing written")
print()
print("A draft WAS created, though - it is sitting in durable state, not on the site:")
print("  pending item ids:", list(state.load()["pending"]))

## Step 6 — Reject, then approve

Rejection first, so you can see that saying no really does leave the website alone.


In [ ]:
rejected = runtime.resume(result.run_id, config, approved=False)
print("Rejected run status :", rejected.status)
print("Website file exists :", target.exists())
print("Events at the end   :", [e["event"] for e in rejected.events[-3:]])

In [ ]:
# Now run the tick again and approve it. This is the only cell in the notebook
# that changes a file on disk, and it does so only because we said approved=True.
second = runtime.run(config, clean.item_id)
print("Paused again on:", second.pending_action["tool"])

approved = runtime.resume(second.run_id, config, approved=True)
print("Status after approval:", approved.status)
print("Tool result          :", approved.output)
print()
print("Website file exists  :", target.exists())
print("--- content/updates.md ---")
print(target.read_text(encoding="utf-8"))

## Step 7 — Change detection over several ticks

The cached source holds two items. A scheduler running daily should work through the
backlog and then go quiet.


In [ ]:
def one_tick(label):
    """Exactly what a scheduler does: find new work, or report that there is none."""
    processed = set(state.load()["processed_ids"])
    todo = [i for i in source.fetch() if i.item_id not in processed]
    print(f"{label:<10} processed={sorted(processed)}")
    if not todo:
        print(f"{'':<10} -> no_change (nothing new; no model call, no run)")
        return None
    run = runtime.run(config, todo[0].item_id)
    print(f"{'':<10} -> new item {todo[0].item_id}: {run.status} on {run.pending_action['tool']}")
    return run

tick2 = one_tick("tick 2")
runtime.resume(tick2.run_id, config, approved=True)
tick3 = one_tick("tick 3")

print()
print("--- final website ---")
print(target.read_text(encoding="utf-8"))

## Step 8 — A poisoned source is refused before anything is drafted

External text is evidence, never instructions. This fixture contains a real-looking
update with "ignore all previous instructions" buried inside it.


In [ ]:
poisoned_source = CachedJSONSource(PROJECT_ROOT / "data" / "poisoned_website_updates.json")
poisoned_registry = build_website_registry(
    poisoned_source, proposer, guardrails, JSONStateStore(RUN_ROOT / "poisoned_state.json"))
poisoned_runtime = HarnessRuntime(poisoned_registry, MockModel(), events)

bad_item = poisoned_source.fetch()[0]
print("Poisoned summary:", bad_item.summary[:90], "...")
print()

blocked = poisoned_runtime.run(website_agent_config(), bad_item.item_id)
print("Run status:", blocked.status)
print("Reason    :", blocked.output)
print()
for event in blocked.events:
    if event["event"] in {"policy_decision", "tool_failed", "run_failed"}:
        print(f"  {event['event']:<16}", event["details"])
print()
print("It never reached publish_update, because propose_update refused to draft at all.")
print("Website file still exists?", target.exists(), "(from Step 6/7, unchanged by this)")

## Step 9 — The live paths, both optional

Two independent live options: a real source (public GitHub releases) and a real model
(OpenRouter writing the update text). Both are guarded, both fall back, and neither can
publish anything — the run still stops at `pending_approval`.


In [ ]:
# Live SOURCE. Off by default: set RUN_LIVE_FETCH=1 in your environment to try it.
from mini_harness import GitHubReleaseSource

RUN_LIVE_FETCH = os.getenv("RUN_LIVE_FETCH") == "1"
print("RUN_LIVE_FETCH =", RUN_LIVE_FETCH)

live_items = []
if RUN_LIVE_FETCH:
    try:
        live_items = GitHubReleaseSource("modelcontextprotocol", "python-sdk", limit=3).fetch()
        print("Fetched", len(live_items), "public releases:")
        for item in live_items:
            print(f"  {item.item_id:<14} {item.title}")
        print("Same UpdateItem contract as the cached source, so nothing downstream changes.")
    except Exception as exc:                 # noqa: BLE001 - network is optional here
        print("Live fetch failed:", type(exc).__name__, exc)
        print("Falling back to the cached source; the lesson is unchanged.")
else:
    print("Skipped. The cached fixture already exercises every guardrail.")

In [ ]:
# Live MODEL. This is the only path where a model actually writes the update text.
from mini_harness import OpenRouterWebsiteProposer

live_proposer = proposer          # default: the deterministic copy-the-summary proposer
if LIVE:
    try:
        live_proposer = OpenRouterWebsiteProposer()
        sample = live_proposer(clean)
        print("Live proposal heading:", sample.heading)
        print("Live proposal body   :", sample.body[:200], "...")
        print()
        print("Compare with the deterministic version, which just copies the summary:")
        print("  ", deterministic_proposer(clean).body[:120], "...")
    except Exception as exc:                 # noqa: BLE001 - one failure must not stop the class
        print("Live proposer unavailable:", type(exc).__name__, exc)
        print("Falling back to deterministic_proposer.")
        live_proposer = proposer
else:
    print("MOCK mode: using deterministic_proposer, which copies the source summary verbatim.")
    print("No model is involved on this path - be honest about that when you present it.")

print()
print("Whichever proposer is used, the guardrails and the approval pause are identical.")
print("That is the point: the model writes text; Python decides what happens to it.")

## Step 10 — Where the scheduler fits

`run_website_agent.py` is the entry point a scheduler calls. It contains no business
logic: it finds new work, starts one bounded harness run, and stops at the approval.


In [ ]:
script = (PROJECT_ROOT / "run_website_agent.py").read_text(encoding="utf-8")
body = [line for line in script.splitlines() if line.startswith(("RUN_ROOT", "result =", "runtime ="))]
print("Key lines from run_website_agent.py:")
for line in body:
    print("  ", line)
print()
print("Run it from a terminal with:  python run_website_agent.py")
print("Schedule it with cron, Windows Task Scheduler or a CI schedule.")
print("The scheduler decides WHEN. It never decides WHETHER - policy and a human do.")

### Try it yourself

Predict: if you re-classified `publish_update` as `write` instead of `external`, what
would this capstone do differently?


In [ ]:
# --- Worked solution ---
# Risk levels are not labels; they are the control. Re-classify the tool and the
# entire human checkpoint disappears - so we do this on a THROWAWAY site folder.
from mini_harness import ToolRegistry, ToolSpec

throwaway_site = RUN_ROOT / "throwaway_site"
throwaway = build_website_registry(
    source, proposer, WebsiteGuardrails(throwaway_site, {"github.com"}),
    JSONStateStore(RUN_ROOT / "throwaway_state.json"))

# Rebuild the registry with publish_update downgraded from external to write.
downgraded = ToolRegistry()
for spec in throwaway.discover():
    risk = "write" if spec.name == "publish_update" else spec.risk
    downgraded.register(ToolSpec(spec.name, spec.description, spec.input_schema, risk),
                        throwaway.get(spec.name).function)

print("publish_update risk is now:", downgraded.get("publish_update").spec.risk)
print("Policy decision            :", decide(config, downgraded.get("publish_update").spec))
print()

runaway = HarnessRuntime(downgraded, MockModel(), EventStore())
outcome = runaway.run(config, clean.item_id)
print("Run status:", outcome.status, "  <- it completed; nobody was asked")
print("Throwaway site written    :", (throwaway_site / "content" / "updates.md").exists())
print()
print("One word in one ToolSpec removed the human from the loop entirely.")
print("Classifying a tool's risk honestly is the single highest-leverage safety decision")
print("in this whole harness - and it is a decision only a person can make.")

## Required live observation

Choose one bounded live observation: fetch up to three public releases or obtain one OpenRouter update proposal. Stop before approval. The cached source and captured trace are the outage fallback.


### Checkpoint

**1. The scheduler runs this every day at 06:00. Does that make it agentic?**

<details><summary>Show answer</summary>

No. The schedule is plain automation - a timer starting a process. The agentic part is deciding that a particular source item is new and worth a website change, and proposing a specific patch for it. Policy then decides whether that proposal may proceed, and in this design it always pauses before publishing.

</details>

**2. In mock mode the proposal text is copied from the source. Is the capstone still real?**

<details><summary>Show answer</summary>

The workflow is real: real files, real durable state, real guardrails, real policy and a real approval gate. What is not real is the *writing* - `deterministic_proposer` copies the summary and makes no model call. Step 9 swaps in `OpenRouterWebsiteProposer`, and note that not one guardrail or policy decision changes when it does.

</details>

### Recap

- Limitation: a workflow that touches real files and a real website is exactly where 'the model decided to' stops being an acceptable explanation.
- Layer added: the workflow's two actions registered as harness tools with honest risk levels, driven by the runtime, gated by policy, recorded as events and paused on a checkpoint - with domain guardrails inside each tool.
- Evidence: `propose_update` allowed and `publish_update` paused; the website file appeared only after an explicit approval; a poisoned source was refused before drafting; and downgrading one risk level removed the human gate entirely.
